In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
# from loader.Vocab import GloVe

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
# from loader.Vocab import GloVe

# 1) Thiết lập chung
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Load Inception-ResNet-V2 (pretrained trên ImageNet)
inception = timm.create_model('inception_resnet_v2', pretrained=True, num_classes=0)
inception.eval().to(DEVICE)

# 3) Load I3D (pretrained trên Kinetics-400) thông qua torch.hub
#    Model này sẽ tự động tải weights lần đầu
i3d = torch.hub.load('facebookresearch/pytorchvideo', 'i3d_r50', pretrained=True)
#    Loại bỏ lớp phân loại cuối để lấy feature map
if hasattr(i3d, 'blocks'):
    i3d.blocks[-1] = nn.Identity()
else:
    i3d.fc = nn.Identity()
i3d.eval().to(DEVICE)

# # 4) Load GloVe vectors tự động qua torchtext
# glove = GloVe(name='6B', dim=300)

# 5) Các transform cho Inception
img_size = 224
img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# 6) Hàm sample frame ở fps nhất định
def sample_frames(video_path, target_fps):
    cap = cv2.VideoCapture(video_path)
    orig_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    interval = max(1, int(round(orig_fps / target_fps)))
    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        idx += 1
    cap.release()
    return frames

# 7) Hàm extract image features từ list các frame
def extract_image_features(frames):
    feats = []
    with torch.no_grad():
        for img in frames:
            x = img_transform(img).unsqueeze(0).to(DEVICE)
            f = inception(x)            # 2048-D
            feats.append(f.cpu().numpy().squeeze())
    return np.stack(feats, axis=0) if feats else np.zeros((0, inception.num_features), dtype=np.float32)

# 8) Resample cố định về N frame features
def resample_fixed(frames_feats, N):
    T, D = frames_feats.shape
    if T == 0:
        return np.zeros((N, D), dtype=np.float32)
    idxs = np.linspace(0, T - 1, N).astype(int)
    return frames_feats[idxs]

# 9) Dataset và DataLoader cho I3D (clip 64 frame, stride=5)
class VideoClipDataset(Dataset):
    def __init__(self, video_path, clip_len=64, stride=5, sample_fps=25):
        self.frames = sample_frames(video_path, sample_fps)
        self.clip_len = clip_len
        self.stride = stride

    def __len__(self):
        return max(0, (len(self.frames) - self.clip_len) // self.stride + 1)

    def __getitem__(self, idx):
        start = idx * self.stride
        clip = self.frames[start:start + self.clip_len]
        tensor_clip = torch.stack([img_transform(img) for img in clip], dim=1)
        return tensor_clip

# 10) Hàm extract motion features
def extract_motion_features(video_path, sample_fps):
    ds = VideoClipDataset(video_path, clip_len=64, stride=5, sample_fps=sample_fps)
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=4)
    all_feats = []
    with torch.no_grad():
        for clips in loader:
            clips = clips.to(DEVICE)
            f = i3d(clips)
            all_feats.append(f.cpu().numpy())
    return np.concatenate(all_feats, axis=0) if all_feats else np.zeros((0, i3d.blocks[-1].out_features if hasattr(i3d, 'blocks') else i3d.fc.in_features), dtype=np.float32)


# 12) Lấy GloVe embedding trung bình cho label
def get_label_embedding(label):
    words = label.lower().split()
    vecs = [glove[w] for w in words if w in glove.stoi]
    if not vecs:
        return np.zeros(glove.dim, dtype=np.float32)
    return np.mean(vecs, axis=0)

# 13) Pipeline tổng hợp cho một video
def process_video(video_path, dataset='MSVD', category_label=None):
    fps_in = 5 if dataset=='MSVD' else 3
    fps_motion = 25 if dataset=='MSVD' else 15

    # image features
    frames1 = sample_frames(video_path, fps_in)
    img_feats = extract_image_features(frames1)
    img_feats = resample_fixed(img_feats, 50 if dataset=='MSVD' else 60)

    # motion features
    mov_feats = extract_motion_features(video_path, fps_motion)

    # # label embedding
    # label_emb = get_label_embedding(category_label) if category_label else None

    # return {'image_feats': img_proj, 'motion_feats': mov_proj, 'label_emb': label_emb}
    return {'image_feats': img_feats, 'motion_feats': mov_feats, 'label_emb': None}

# Ví dụ sử dụng
if __name__ == '__main__':
    video_file = '/content/v_GGSY1Qvo990.mp4'
    res = process_video(video_file, dataset='MSVD', category_label='sports')
    print('Image:', res['image_feats'].shape)
    print('Motion:', res['motion_feats'].shape)
    # print('Label:', None if res['label_emb'] is None else res['label_emb'].shape)


In [ ]:
# Save output torch array to file
## Save image features to file
torch.save(torch.tensor(res['image_feats']), 'image_features.pt')
## Save motion features to file
torch.save(torch.tensor(res['motion_feats']), 'motion_features.pt')

In [11]:
# Load the saved features
image_features = torch.load('image_features.pt')
motion_features = torch.load('motion_features.pt')
# Reshape to 3D tensor, the first dimension is 1
image_features = image_features.unsqueeze(0)
motion_features = motion_features.unsqueeze(0)

print("Image features shape:", image_features.shape)
print("Motion features shape:", motion_features.shape)

Image features shape: torch.Size([1, 50, 1536])
Motion features shape: torch.Size([1, 50, 2048])


tensor([[0.3030, 0.1537, 0.8104,  ..., 0.0026, 0.3686, 0.8653],
        [0.2506, 0.1908, 0.6986,  ..., 0.0047, 0.5023, 1.0355],
        [0.2222, 0.0725, 0.9149,  ..., 0.0221, 0.3099, 0.5430],
        ...,
        [0.9263, 0.1024, 1.1582,  ..., 0.4720, 0.5155, 1.2529],
        [0.5638, 0.2409, 1.0505,  ..., 0.3768, 0.4696, 0.8724],
        [0.4175, 0.1349, 0.9237,  ..., 0.5985, 0.5858, 0.8521]])

# NEW PAGE

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import timm
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import roi_align
# from loader.Vocab import GloVe

# 1) Thiết lập chung
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Load Inception-ResNet-V2 (pretrained)
inception = timm.create_model('inception_resnet_v2', pretrained=True, num_classes=0)
inception.eval().to(DEVICE)

# 3) Load I3D (pretrained) cho motion features
i3d = torch.hub.load('facebookresearch/pytorchvideo', 'i3d_r50', pretrained=True)
if hasattr(i3d, 'blocks'):
    i3d.blocks[-1] = nn.Identity()
else:
    i3d.fc = nn.Identity()
i3d.eval().to(DEVICE)

# 4) Load Mask R-CNN (pretrained) cho object detection & ROI features
maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
maskrcnn.eval().to(DEVICE)

# 5) Embedding for object relationships (TransE-like)
NUM_CLASSES = 91  # COCO classes
torch.manual_seed(0)
rel_embedding = nn.Embedding(NUM_CLASSES, 300).to(DEVICE)

# 6) Projections: image→512, motion→512, object→1028, relation→300 (embedding)
proj_image  = nn.Linear(2048, 512).to(DEVICE)
proj_motion = nn.Linear(2048, 512).to(DEVICE)
proj_object = nn.Linear(256, 1028).to(DEVICE)

# 7) Transform cho Inception & Mask R-CNN
img_size = 299
img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# 8) Hàm sample frame ở fps nhất định
def sample_frames(video_path, target_fps):
    cap = cv2.VideoCapture(video_path)
    orig_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    interval = max(1, int(round(orig_fps / target_fps)))
    frames, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % interval == 0:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        idx += 1
    cap.release()
    return frames

# 9) Extract image features
def extract_image_features(frames):
    feats = []
    with torch.no_grad():
        for img in frames:
            x = img_transform(img).unsqueeze(0).to(DEVICE)
            f = inception(x)  # [1,2048]
            feats.append(f.cpu().numpy().squeeze())
    return np.stack(feats, axis=0) if feats else np.zeros((0, 2048), dtype=np.float32)

# 10) Resample cố định N frames
import numpy as np
def resample_fixed(frames_feats, N):
    T, D = frames_feats.shape
    if T == 0:
        return np.zeros((N, D), dtype=np.float32)
    idxs = np.linspace(0, T-1, N).astype(int)
    return frames_feats[idxs]

# 11) Dataset cho motion (I3D clips)
class VideoClipDataset(Dataset):
    def __init__(self, video_path, clip_len=64, stride=5, sample_fps=25):
        self.frames = sample_frames(video_path, sample_fps)
        self.clip_len, self.stride = clip_len, stride
    def __len__(self):
        return max(0, (len(self.frames)-self.clip_len)//self.stride + 1)
    def __getitem__(self, idx):
        start = idx*self.stride
        clip = self.frames[start:start+self.clip_len]
        tensor_clip = torch.stack([img_transform(img) for img in clip], dim=1)
        return tensor_clip

# 12) Extract motion features
def extract_motion_features(video_path, sample_fps):
    ds = VideoClipDataset(video_path, clip_len=64, stride=5, sample_fps=sample_fps)
    loader = DataLoader(ds, batch_size=2, shuffle=False, num_workers=2)
    feats = []
    with torch.no_grad():
        for clips in loader:
            clips = clips.to(DEVICE)
            f = i3d(clips)  # [B,2048,T,H,W]
            f = f.mean(dim=[2,3,4])  # -> [B,2048]
            feats.append(f.cpu().numpy())
    return np.concatenate(feats,0) if feats else np.zeros((0,2048), dtype=np.float32)

# 13) Extract object & relation features từ nhiều frame
#    Trả về obj_feats: (N_frames, 1028), rel_feats: (N_frames, 300)
def extract_object_rel_sequence(frames, N_frames):
    T = len(frames)
    if T == 0:
        return np.zeros((N_frames, 1028), dtype=np.float32), np.zeros((N_frames, 300), dtype=np.float32)
    # Lấy chỉ N_frames ở khoảng đồng đều
    idxs = np.linspace(0, T-1, N_frames).astype(int)
    obj_list, rel_list = [], []
    with torch.no_grad():
        for i in idxs:
            img = frames[i]
            x = img_transform(img).to(DEVICE)
            out = maskrcnn([x])[0]
            boxes, labels = out['boxes'], out['labels']
            feats_map = maskrcnn.backbone(x.unsqueeze(0))
            feats_tensor = list(feats_map.values())[0]  # [1,256,H,W]
            roi_feats = roi_align(feats_tensor, [boxes], output_size=(1,1))  # [N_obj,256,1,1]
            roi_feats = roi_feats.view(-1,256)
            if roi_feats.numel() == 0:
                obj = torch.zeros((1,1028), device=DEVICE)
                rel = torch.zeros((1,300), device=DEVICE)
            else:
                o = proj_object(roi_feats)            # [N_obj,1028]
                obj = o.mean(dim=0, keepdim=True)    # [1,1028]
                r = rel_embedding(labels.to(DEVICE)) # [N_obj,300]
                rel = r.mean(dim=0, keepdim=True)    # [1,300]
            obj_list.append(obj.cpu().numpy().squeeze())
            rel_list.append(rel.cpu().numpy().squeeze())
    return np.stack(obj_list, axis=0), np.stack(rel_list, axis=0)

# 14) Pipeline tổng hợp cho video
def process_video(video_path, dataset='MSVD'):
    # a) Image features
    fps_in = 5 if dataset=='MSVD' else 3
    img_frames = sample_frames(video_path, fps_in)
    img_feats = extract_image_features(img_frames)
    img_feats = resample_fixed(img_feats, 50 if dataset=='MSVD' else 60)
    img_proj = proj_image(torch.from_numpy(img_feats).to(DEVICE)).cpu().numpy()

    # b) Motion features
    fps_m = 25 if dataset=='MSVD' else 15
    mot_feats = extract_motion_features(video_path, fps_m)
    mot_feats = resample_fixed(mot_feats, 50 if dataset=='MSVD' else 60)
    mot_proj = proj_motion(torch.from_numpy(mot_feats).to(DEVICE)).cpu().numpy()

    # c) Object & relation features sequence
    N = 50 if dataset=='MSVD' else 60
    obj_feats, rel_feats = extract_object_rel_sequence(img_frames, N)

    return {
        'image_feats': img_proj,    # [N,512]
        'motion_feats': mot_proj,   # [N,512]
        'object_feats': obj_feats,  # [N,1028]
        'rel_feats': rel_feats      # [N,300]
    }

# Ví dụ sử dụng
if __name__ == '__main__':
    video_file = '/path/to/video.mp4'
    res = process_video(video_file, dataset='MSVD')
    for k,v in res.items(): print(f"{k}: {v.shape}")
